# Waktu Proses ONNX

Di notebook ini, kami akan menunjukkan cara menggunakan ONNX Runtime untuk mempercepat inferensi model yang dilatih di PyTorch. Selain itu, kami akan menggunakan ONNX untuk mengkuantisasi model hingga presisi int8 guna lebih meningkatkan kinerja dengan mengurangi jejak memori. Kami akan melatih model sederhana pada kumpulan data MNIST dan kemudian mengonversinya ke format ONNX. Kami kemudian akan menggunakan ONNX Runtime untuk mempercepat inferensi model. Terakhir, kami akan mengkuantisasi model hingga presisi int8

## Atur Waktu Proses ONNX

Pertama, instal torch, torchvision, onnx dan onnxruntime. Kemudian, impor modul yang diperlukan

In [1]:
%pip install torch torchvision
%pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.6 MB/s eta 0:00:00


In [2]:
# Library PyTorch untuk deep learning
import torch  # Library utama PyTorch untuk tensor dan operasi jaringan neural
import torch.nn as nn  # Modul PyTorch untuk membangun lapisan jaringan neural
import torch.nn.functional as F  # Berisi fungsi-fungsi aktivasi dan operasi lainnya
import torch.optim as optim  # Modul untuk algoritma optimisasi seperti SGD atau Adam

# Library torchvision untuk dataset dan transformasi gambar
from torchvision import datasets, transforms  # Untuk memuat dataset dan melakukan preprocessing

# Library PyTorch untuk quantization
import torch.quantization  # Fitur untuk mengonversi model ke format quantized (INT8)

# Library pathlib untuk manipulasi path file secara lintas platform
import pathlib

# Library numpy untuk operasi numerik
import numpy as np

# Library PyTorch untuk ekspor model ke format ONNX
import torch.onnx

# Library ONNX untuk memvalidasi dan memodifikasi model ONNX
import onnx

# Library ONNX Runtime untuk menjalankan model ONNX
import onnxruntime

# Fitur untuk quantization dalam ONNX Runtime
from onnxruntime.quantization import (
    quantize_dynamic,  # Quantization dinamis (aktivasi tetap FP32, bobot menjadi INT8)
    quantize_static,   # Quantization statis (kalibrasi digunakan untuk aktivasi dan bobot INT8)
    CalibrationDataReader,  # Kelas dasar untuk membaca data kalibrasi
    QuantType           # Opsi tipe quantization (misalnya, QuantType.QUInt8 untuk INT8)
)


## Model Kereta Api

Kami akan melatih model CNN sederhana pada dataset MNIST.

In [3]:
# Transformasi data: mengubah gambar ke tensor dan menormalisasi
transform = transforms.Compose([
    transforms.ToTensor(),  # Mengubah gambar menjadi tensor
    transforms.Normalize((0.1307,), (0.3081,))  # Normalisasi dengan mean=0.1307 dan std=0.3081
])

# Memuat dataset MNIST untuk pelatihan dan pengujian
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)  # Dataset pelatihan
test_dataset = datasets.MNIST('./data', train=False, transform=transform)  # Dataset pengujian

# Mendefinisikan arsitektur jaringan neural sederhana
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=12, kernel_size=3)  # Lapisan konvolusi
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Lapisan pooling
        self.fc = nn.Linear(12 * 13 * 13, 10)  # Lapisan fully connected (linear)

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)  # Mengubah input menjadi ukuran (batch_size, 1, 28, 28)
        x = F.relu(self.conv1(x))  # Aktivasi ReLU setelah lapisan konvolusi
        x = self.pool(x)  # Pooling untuk mengurangi dimensi spasial
        x = x.view(x.size(0), -1)  # Flatten tensor sebelum masuk ke lapisan fully connected
        x = self.fc(x)  # Lapisan fully connected
        output = F.log_softmax(x, dim=1)  # Fungsi log-softmax untuk output probabilitas log
        return output


# Membuat DataLoader untuk memuat data dalam batch
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32)  # DataLoader pelatihan
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32)  # DataLoader pengujian

# Menentukan perangkat (CPU)
device = "cpu"

# Jumlah epoch untuk pelatihan
epochs = 1

# Membuat model dan memindahkannya ke perangkat
model = Net().to(device)

# Optimizer menggunakan Adam
optimizer = optim.Adam(model.parameters())

# Mode pelatihan untuk model
model.train()

# Loop pelatihan
for epoch in range(1, epochs + 1):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  # Memindahkan data ke perangkat
        optimizer.zero_grad()  # Mengatur gradien ke nol
        output = model(data)  # Melakukan forward pass
        loss = F.nll_loss(output, target)  # Menghitung loss (Negative Log-Likelihood)
        loss.backward()  # Backpropagation
        optimizer.step()  # Memperbarui parameter model
        print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
            epoch, batch_idx * len(data), len(train_loader.dataset),
            100. * batch_idx / len(train_loader), loss.item()))

# Menyimpan model ke direktori `./onnx_models`
MODEL_DIR = pathlib.Path("./onnx_models")
MODEL_DIR.mkdir(exist_ok=True)  # Membuat direktori jika belum ada
torch.save(model.state_dict(), MODEL_DIR / "original_model.p")  # Menyimpan bobot model


Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9.91M/9.91M [00:00<00:00, 57.1MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28.9k/28.9k [00:00<00:00, 2.04MB/s]

Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw



...

100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4.54k/4.54k [00:00<00:00, 7.79MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



Train Epoch: 1 [0/60000 (0%)]	Loss: 2.330969
Train Epoch: 1 [32/60000 (0%)]	Loss: 2.370600
Train Epoch: 1 [64/60000 (0%)]	Loss: 2.184278
Train Epoch: 1 [96/60000 (0%)]	Loss: 2.031038
Train Epoch: 1 [128/60000 (0%)]	Loss: 2.078423
Train Epoch: 1 [160/60000 (0%)]	Loss: 2.083939
Train Epoch: 1 [192/60000 (0%)]	Loss: 1.688972
Train Epoch: 1 [224/60000 (0%)]	Loss: 1.777475
Train Epoch: 1 [256/60000 (0%)]	Loss: 1.891660
Train Epoch: 1 [288/60000 (0%)]	Loss: 1.526311
Train Epoch: 1 [320/60000 (1%)]	Loss: 1.654433
Train Epoch: 1 [352/60000 (1%)]	Loss: 1.453821
Train Epoch: 1 [384/60000 (1%)]	Loss: 1.422168
Train Epoch: 1 [416/60000 (1%)]	Loss: 1.356842
Train Epoch: 1 [448/60000 (1%)]	Loss: 1.251338
Train Epoch: 1 [480/60000 (1%)]	Loss: 1.603687
Train Epoch: 1 [512/60000 (1%)]	Loss: 1.367763
Train Epoch: 1 [544/60000 (1%)]	Loss: 1.115706
Train Epoch: 1 [576/60000 (1%)]	Loss: 1.355937
Train Epoch: 1 [608/60000 (1%)]	Loss: 1.381475
Train Epoch: 1 [640/60000 (1%)]	Loss: 1.308299
Train Epoch: 1 [67

## Ekspor ke ONNX

Setelah pelatihan, ekspor model ke format ONNX.

In [4]:
# Mengambil batch pertama dari train_loader sebagai input contoh
x, _ = next(iter(train_loader))

# Mengekspor model PyTorch ke format ONNX
torch.onnx.export(
    model,                      # Model PyTorch yang akan diekspor
    x,                          # Input contoh untuk menentukan bentuk input
    MODEL_DIR / "mnist_model.onnx",  # Path file untuk menyimpan model ONNX
    export_params=True,         # Mengekspor semua parameter (bobot) dari model
    opset_version=10,           # Versi opset ONNX yang digunakan (10 kompatibel untuk banyak aplikasi)
    do_constant_folding=True,   # Mengoptimalkan subgrafik yang bersifat konstan
    input_names=['input'],      # Nama untuk tensor input
    output_names=['output'],    # Nama untuk tensor output
    dynamic_axes={              # Menentukan dimensi dinamis untuk batch size
        'input': {0: 'batch_size'},   # Dimensi pertama (0) pada input adalah ukuran batch
        'output': {0: 'batch_size'}  # Dimensi pertama (0) pada output adalah ukuran batch
    }
)


## Jalankan Inferensi dan Uji Kemiripan

Selanjutnya, validasi model yang dikonversi dengan menjalankan inferensi dan membandingkan hasilnya dengan model PyTorch.

In [5]:
# Melakukan prediksi menggunakan model PyTorch
torch_out = model(x)

# Memuat model ONNX
onnx_model = onnx.load(MODEL_DIR / "mnist_model.onnx")

# Memvalidasi model ONNX untuk memastikan strukturnya valid
onnx.checker.check_model(onnx_model)

# Membuat sesi inferensi ONNX Runtime dengan model yang diekspor
ort_session = onnxruntime.InferenceSession(
    MODEL_DIR / "mnist_model.onnx",
    providers=["CPUExecutionProvider"]  # Menggunakan eksekusi di CPU
)

# Fungsi untuk mengubah tensor PyTorch menjadi array NumPy
def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

# Menyiapkan input untuk ONNX Runtime
ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(x)}

# Melakukan inferensi menggunakan ONNX Runtime
ort_outs = ort_session.run(None, ort_inputs)

# Membandingkan hasil prediksi dari ONNX Runtime dan PyTorch
np.testing.assert_allclose(
    to_numpy(torch_out),    # Output dari PyTorch
    ort_outs[0],            # Output dari ONNX Runtime
    rtol=1e-03,             # Relatif toleransi kesalahan
    atol=1e-05              # Absolut toleransi kesalahan
)

# Menampilkan pesan jika hasil PyTorch dan ONNX sesuai
print("Exported model has been tested with ONNXRuntime, and the result looks good!")


Exported model has been tested with ONNXRuntime, and the result looks good!


## Quantization

### Kuantisasi Dinamis

Kuantisasi dinamis menghitung parameter yang akan dikuantisasi untuk aktivasi secara dinamis. Perhitungan ini meningkatkan akurasi model namun juga meningkatkan biaya inferensi.

In [6]:
!python -m onnxruntime.quantization.preprocess --input {MODEL_DIR / "mnist_model.onnx"} --output {MODEL_DIR / "mnist_model_processed.onnx"}

In [7]:
# Lokasi model ONNX yang diekspor (FP32)
model_fp32 = MODEL_DIR / "mnist_model_processed.onnx"

# Lokasi untuk menyimpan model yang telah diquantisasi (INT8)
model_quant = MODEL_DIR / "mnist_model_quant.onnx"

# Melakukan quantization dinamis pada model ONNX
quantized_model = quantize_dynamic(
    model_fp32,          # Model ONNX dalam format FP32 (float 32-bit)
    model_quant,         # Nama file output untuk model quantized
    weight_type=QuantType.QUInt8  # Jenis quantization: bobot diubah ke unsigned INT8
)


### Bandingkan Ukuran

Mari kita bandingkan ukuran model aslinya, model terkuantisasi

In [8]:
%ls -lh {MODEL_DIR}

total 280K
-rw-r--r-- 1 root root 82K Jan  3 15:14 mnist_model.onnx
-rw-r--r-- 1 root root 82K Jan  3 15:14 mnist_model_processed.onnx
-rw-r--r-- 1 root root 26K Jan  3 15:14 mnist_model_quant.onnx
-rw-r--r-- 1 root root 82K Jan  3 15:14 original_model.p


### Bandingkan Akurasi

Mari kita bandingkan keakuratan model onnx yang dikonversi dan model terkuantisasi. Keakuratan model terkuantisasi harus mendekati model aslinya

In [9]:
# Fungsi untuk menguji model ONNX menggunakan ONNX Runtime
def test_onnx(model_name, data_loader):
    # Memuat model ONNX
    onnx_model = onnx.load(model_name)

    # Memeriksa validitas model ONNX
    onnx.checker.check_model(onnx_model)

    # Membuat sesi inferensi dengan ONNX Runtime
    ort_session = onnxruntime.InferenceSession(model_name)

    test_loss = 0  # Inisialisasi variabel untuk akumulasi loss
    correct = 0    # Inisialisasi variabel untuk menghitung prediksi yang benar

    # Iterasi melalui data loader
    for data, target in data_loader:
        # Menyiapkan input untuk ONNX Runtime
        ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(data)}

        # Melakukan inferensi
        output = ort_session.run(None, ort_inputs)[0]

        # Mengubah output ONNX Runtime ke tensor PyTorch
        output = torch.from_numpy(output)

        # Menghitung loss batch dan menjumlahkan
        test_loss += F.nll_loss(output, target, reduction='sum').item()

        # Mendapatkan prediksi dengan probabilitas tertinggi
        pred = output.argmax(dim=1, keepdim=True)

        # Menjumlahkan jumlah prediksi yang benar
        correct += pred.eq(target.view_as(pred)).sum().item()

    # Menghitung rata-rata loss
    test_loss /= len(data_loader.dataset)

    # Menghitung akurasi model
    return 100. * correct / len(data_loader.dataset)

# Menguji akurasi model ONNX asli (FP32)
acc = test_onnx(MODEL_DIR / "mnist_model.onnx", test_loader)
print(f"Accuracy of the original model is {acc}%")

# Menguji akurasi model ONNX yang diquantisasi (INT8)
qacc = test_onnx(MODEL_DIR / "mnist_model_quant.onnx", test_loader)
print(f"Accuracy of the quantized model is {qacc}%")


Accuracy of the original model is 96.47%
Accuracy of the quantized model is 96.53%


## Kuantisasi Statis

Untuk metode kuantisasi statis, parameter dikuantisasi terlebih dahulu menggunakan dataset kalibrasi. Metode ini lebih cepat dibandingkan kuantisasi dinamis namun akurasinya lebih rendah. Oleh karena itu, kumpulan data kalbrasi perlu dibuat menggunakan kelas `CalibrationDataReader`.

In [10]:
# Kelas DataReader untuk kalibrasi statis dengan ONNX Runtime
class QuantDR(CalibrationDataReader):
    def __init__(self, torch_data_loader, input_name):
        self.torch_data_loader = torch_data_loader  # DataLoader dari PyTorch
        self.input_name = input_name  # Nama input model ONNX
        self.datasize = len(torch_data_loader)  # Jumlah batch dalam DataLoader
        self.enum_data = iter(torch_data_loader)  # Iterator untuk batch data

    # Fungsi untuk mengubah tensor PyTorch menjadi array NumPy
    def to_numpy(self, tensor):
        return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

    # Mendapatkan batch berikutnya untuk kalibrasi
    def get_next(self):
        batch = next(self.enum_data, None)  # Mengambil batch berikutnya
        if batch is not None:
            return {self.input_name: self.to_numpy(batch[0])}  # Hanya mengembalikan data input
        else:
            return None  # Tidak ada data lagi

    # Mengatur ulang iterator untuk batch data
    def rewind(self):
        self.enum_data = iter(self.torch_data_loader)

# Membuat instance DataReader untuk kalibrasi
calibration_data = QuantDR(train_loader, ort_session.get_inputs()[0].name)

# Lokasi untuk menyimpan model hasil quantization statis
model__static_quant = MODEL_DIR / "mnist_model_static_quant.onnx"

# Melakukan quantization statis pada model ONNX
static_quant_model = quantize_static(
    model_fp32,                    # Model ONNX asli (FP32)
    model__static_quant,           # Lokasi untuk menyimpan model hasil quantization
    calibration_data,              # Data kalibrasi
    weight_type=QuantType.QInt8    # Jenis quantization: bobot menjadi signed INT8
)


### Bandingkan Ukuran

Mari kita bandingkan ukuran model asli dan model terkuantisasi

In [11]:
%ls -lh {MODEL_DIR}

total 308K
-rw-r--r-- 1 root root 82K Jan  3 15:14 mnist_model.onnx
-rw-r--r-- 1 root root 82K Jan  3 15:14 mnist_model_processed.onnx
-rw-r--r-- 1 root root 26K Jan  3 15:14 mnist_model_quant.onnx
-rw-r--r-- 1 root root 25K Jan  3 15:15 mnist_model_static_quant.onnx
-rw-r--r-- 1 root root 82K Jan  3 15:14 original_model.p


### Bandingkan Akurasi

Mari kita bandingkan keakuratan model onnx yang dikonversi dan model terkuantisasi. Keakuratan model terkuantisasi harus mendekati model aslinya

In [12]:
static_qacc = test_onnx(model__static_quant, test_loader)
print(f"Accuracy of the static quantized model is {static_qacc}%")

Accuracy of the static quantized model is 96.51%
